In [1]:
# %%
from pathlib import Path
import os
import re
import json
import math
from collections import defaultdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

import spacy
import scispacy  # noqa: F401

c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()
PARSED_DIR = PROJECT_ROOT / "data" / "parsed"
INDEX_DIR = PROJECT_ROOT / "data" / "index"

IN_CHUNKS = PARSED_DIR / "pmc_retrieval_candidates.parquet"
IN_CONCEPTS = PARSED_DIR / "graph_concepts.parquet"
IN_EDGE_EVID = PARSED_DIR / "graph_edge_evidence.parquet"
IN_EDGES_SC = PARSED_DIR / "graph_edges_symptom_condition.parquet"

FAISS_INDEX = INDEX_DIR / "chunks.faiss"
LOOKUP_PARQUET = INDEX_DIR / "chunk_lookup.parquet"
META_JSON = INDEX_DIR / "chunk_meta.json"

OUT_LOG = PARSED_DIR / "query_logs.jsonl"

for p in [
    IN_CHUNKS, IN_CONCEPTS, IN_EDGE_EVID, IN_EDGES_SC,
    FAISS_INDEX, LOOKUP_PARQUET, META_JSON
]:
    assert p.exists(), f"Missing required input: {p}"

In [3]:
# %%
chunks_df = pd.read_parquet(IN_CHUNKS)[
    ["chunk_id", "pmcid", "article_title", "section", "chunk_text", "token_count", "license"]
]

concepts_df = pd.read_parquet(IN_CONCEPTS)
edge_evid_df = pd.read_parquet(IN_EDGE_EVID)
edges_sc_df = pd.read_parquet(IN_EDGES_SC)

lookup_df = pd.read_parquet(LOOKUP_PARQUET)
meta = json.loads(META_JSON.read_text())

print("Chunks:", len(chunks_df))
print("Concepts:", len(concepts_df))
print("Edge evidence:", len(edge_evid_df))
print("Edges S–C:", len(edges_sc_df))

Chunks: 12935
Concepts: 5779
Edge evidence: 609882
Edges S–C: 290910


In [4]:
# %%
EMBED_MODEL = meta["embedding_model"]
embedder = SentenceTransformer(EMBED_MODEL)

index = faiss.read_index(str(FAISS_INDEX))
assert index.d == meta["embedding_dim"]

print("FAISS size:", index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 874.78it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS size: 12935


In [5]:
# %%
# Lightweight NER only — NO UMLS linker
nlp = spacy.load("en_core_sci_sm")

print("Pipeline:", nlp.pipe_names)

Pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [6]:
def normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9\s]", "", s.lower()).strip()

# High-value manual aliases for common user phrasing
ALIASES = {
    "low blood pressure": "hypotension",
    "confusion": "altered mental status",
    "mental confusion": "altered mental status",
    "disorientation": "altered mental status",
    "infection": "infection",
}

symptom_concepts = concepts_df[
    concepts_df["concept_types"].astype(str).str.contains("SYMPTOM")
].copy()

known_symptoms: Dict[str, str] = {}

for _, r in symptom_concepts.iterrows():
    name = r.get("canonical_name")
    if not isinstance(name, str) or not name.strip():
        continue

    canon = normalize_text(name)
    known_symptoms[canon] = r["concept_id"]

    # inject aliases that map to this canonical symptom
    for alias, target in ALIASES.items():
        if target in canon:
            known_symptoms[normalize_text(alias)] = r["concept_id"]

print("Known symptom strings (with aliases):", len(known_symptoms))

Known symptom strings (with aliases): 2641


In [7]:
def extract_symptom_concepts(
    query: str,
    known_symptoms: Dict[str, str],
) -> List[Tuple[str, float, str]]:
    """
    Returns (concept_id, score, surface_text)
    Ensures surface_text is never empty.
    """
    doc = nlp(query)
    hits = {}

    q_norm = normalize_text(query)

    # 1) substring match over full query
    for k, cid in known_symptoms.items():
        if k and k in q_norm:
            hits[cid] = (1.0, k)

    # 2) NER span match (preferred surface form)
    for ent in doc.ents:
        ent_norm = normalize_text(ent.text)
        if ent_norm in known_symptoms:
            cid = known_symptoms[ent_norm]
            hits[cid] = (1.0, ent.text)

    # drop any empty surface texts
    return [
        (cid, sc, txt)
        for cid, (sc, txt) in hits.items()
        if isinstance(txt, str) and txt.strip()
    ]

In [8]:
def graph_first_candidates(
    symptom_ids: List[str],
    top_k: int = 40,
) -> pd.DataFrame:
    """
    Graph-first retrieval with clinical symptom weighting.
    Fever is downweighted because it is non-specific.
    """
    if not symptom_ids:
        return pd.DataFrame(columns=["condition_concept_id", "graph_score", "graph_support"])

    # Clinical symptom weights
    SYMPTOM_WEIGHTS = {
        "UMLS:C0015967": 0.3,  # fever (non-specific)
        "UMLS:C4304283": 1.0,  # hypotension
        "UMLS:C5187432": 1.0,  # altered mental status
    }

    sub = edges_sc_df[edges_sc_df["symptom_concept_id"].isin(symptom_ids)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["condition_concept_id", "graph_score", "graph_support"])

    # Apply symptom weighting
    sub["symptom_weight"] = sub["symptom_concept_id"].map(
        lambda cid: SYMPTOM_WEIGHTS.get(cid, 1.0)
    )

    sub["weighted_graph_score"] = sub["weighted_score"] * sub["symptom_weight"]

    return (
        sub.groupby("condition_concept_id", as_index=False)
        .agg(
            graph_score=("weighted_graph_score", "sum"),
            graph_support=("support_count", "sum"),
        )
        .sort_values(["graph_score", "graph_support"], ascending=False)
        .head(top_k)
    )

In [9]:
# %%
edge_evid_by_chunk = edge_evid_df[[
    "chunk_id", "condition_concept_id", "evidence_score", "pmcid"
]]

def faiss_chunks(query: str, k: int = 30) -> pd.DataFrame:
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q, k)
    out = lookup_df.iloc[idxs[0]].copy()
    out["similarity"] = scores[0]
    return out

def embedding_candidates(query: str, k_chunks=30, k_cond=40):
    chunks = faiss_chunks(query, k_chunks)
    sub = edge_evid_by_chunk[edge_evid_by_chunk["chunk_id"].isin(chunks["chunk_id"])]
    if sub.empty:
        return pd.DataFrame(columns=["condition_concept_id", "emb_score", "emb_support"])

    return (
        sub.groupby("condition_concept_id", as_index=False)
        .agg(
            emb_score=("evidence_score", "sum"),
            emb_support=("pmcid", "nunique"),
        )
        .sort_values(["emb_score", "emb_support"], ascending=False)
        .head(k_cond)
    )

In [10]:
# %%
def log1p(x: float) -> float:
    return math.log1p(max(0.0, x))

def merge_rerank(graph_df, emb_df, use_graph: bool, top_n=5):
    merged = pd.merge(
    graph_df, emb_df,
    on="condition_concept_id",
    how="outer"
)
    merged = merged.infer_objects(copy=False).fillna(0.0)

    # Dynamic weighting based on symptom specificity
    if use_graph:
        W_GRAPH, W_EMB = 0.8, 0.2
    else:
        W_GRAPH, W_EMB = 0.4, 0.6

    merged["final_score"] = (
        W_GRAPH * merged["graph_score"].apply(log1p)
        + W_EMB * merged["emb_score"].apply(log1p)
    )

    return merged.sort_values(
        ["final_score", "graph_support", "emb_support"],
        ascending=False
    ).head(top_n)

In [11]:
# %%
def select_evidence(
    condition_id: str,
    symptom_ids: List[str],
    max_snips=6,
):
    sub = edge_evid_df[edge_evid_df["condition_concept_id"] == condition_id]

    if symptom_ids:
        sub2 = sub[sub["symptom_concept_id"].isin(symptom_ids)]
        if not sub2.empty:
            sub = sub2

    picked = []
    seen = defaultdict(int)

    for _, r in sub.sort_values("evidence_score", ascending=False).iterrows():
        if seen[r["pmcid"]] >= 2:
            continue
        picked.append(r)
        seen[r["pmcid"]] += 1
        if len(picked) >= max_snips:
            break

    return pd.DataFrame(picked)

In [12]:
# %%
name_map = concepts_df.set_index("concept_id")["canonical_name"].to_dict()

def build_context(
    query: str,
    symptoms,
    ranked,
):
    conditions = []

    for _, r in ranked.iterrows():
        ev = select_evidence(r["condition_concept_id"], [s[0] for s in symptoms])
        conditions.append({
            "concept_id": r["condition_concept_id"],
            "label": name_map.get(r["condition_concept_id"]),
            "final_score": float(r["final_score"]),
            "evidence": ev[[
                "pmcid", "article_title", "section", "chunk_id", "snippet", "evidence_score"
            ]].to_dict(orient="records"),
        })

    return {
        "query": query,
        "symptoms": [
            {"concept_id": cid, "label": name_map.get(cid), "mention": txt}
            for cid, _, txt in symptoms
        ],
        "conditions": conditions,
    }

In [13]:
# %%
def deepseek_generate(context: Dict) -> str:
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("Set DEEPSEEK_API_KEY first")

    from openai import OpenAI
    client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

    system = (
        "You are a biomedical literature assistant. "
        "You do not diagnose or give treatment advice. "
        "You explain associations using cited PMC evidence."
    )

    user = (
        "Use ONLY the evidence below. "
        "Explain associations and cite PMCIDs.\n\n"
        f"{json.dumps(context, ensure_ascii=False)}"
    )

    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.2,
    )

    return resp.choices[0].message.content

In [14]:
def run_query(
    query: str,
    generate: bool = False,
):
    # =========================
    # SYMPTOM EXTRACTION
    # =========================
    symptoms = extract_symptom_concepts(query, known_symptoms)

    # =========================
    # DROP OVERLY GENERIC SYMPTOMS
    # =========================
    GENERIC_SYMPTOMS = {
        "UMLS:C5139432",  # infection (too non-specific)
    }

    symptoms = [
        s for s in symptoms
        if s[0] not in GENERIC_SYMPTOMS
    ]

    symptom_ids = [cid for cid, _, _ in symptoms]

    # =========================
    # GRAPH USAGE GATE (FINAL LOGIC)
    # =========================
    # Fever alone is not specific.
    # Hypotension / altered mental status ARE specific.
    SPECIFIC_SYMPTOMS = {
        "UMLS:C4304283",  # hypotension
        "UMLS:C5187432",  # altered mental status
    }

    specific_count = sum(
        1 for cid, _, _ in symptoms
        if cid in SPECIFIC_SYMPTOMS
    )

    USE_GRAPH = specific_count >= 1

    # =========================
    # RETRIEVAL
    # =========================
    graph_df = graph_first_candidates(symptom_ids)
    emb_df = embedding_candidates(query)

    ranked = merge_rerank(
        graph_df,
        emb_df,
        use_graph=USE_GRAPH
    )

    # =========================
    # CONTEXT BUILD
    # =========================
    context = build_context(query, symptoms, ranked)

    out = {
        "context": context,
        "ranked_conditions": ranked.to_dict(orient="records"),
        "symptoms": symptoms,
    }

    # =========================
    # OPTIONAL GENERATION
    # =========================
    if generate:
        out["answer"] = deepseek_generate(context)

    # =========================
    # LOGGING
    # =========================
    with open(OUT_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(out, ensure_ascii=False) + "\n")

    return out

In [15]:
out = run_query(
    "fever, low blood pressure, infection, confusion",
    generate=False
)

print("Symptoms:")
print(out["symptoms"])

print("\nRanked conditions:")
for r in out["ranked_conditions"]:
    print(r)

Symptoms:
[('UMLS:C0015967', 1.0, 'fever'), ('UMLS:C4304283', 1.0, 'low blood pressure')]

Ranked conditions:
{'condition_concept_id': 'UMLS:C0034362', 'graph_score': 0.7886567023057153, 'graph_support': 23.0, 'emb_score': 10.903358629641064, 'emb_support': 3.0, 'final_score': 0.9605360336066628}
{'condition_concept_id': 'UMLS:C3714514', 'graph_score': 0.7240933295506572, 'graph_support': 15.0, 'emb_score': 12.273286333288658, 'emb_support': 3.0, 'final_score': 0.9529117388019791}
{'condition_concept_id': 'UMLS:C0948075', 'graph_score': 0.6051969229632648, 'graph_support': 13.0, 'emb_score': 11.398360423222996, 'emb_support': 3.0, 'final_score': 0.8821100019796504}
{'condition_concept_id': 'UMLS:C2242472', 'graph_score': 0.5979497315194046, 'graph_support': 13.0, 'emb_score': 11.246553554017312, 'emb_support': 3.0, 'final_score': 0.876026022757479}
{'condition_concept_id': 'UMLS:C0205339', 'graph_score': 0.58100756963288, 'graph_support': 12.0, 'emb_score': 11.759403585588798, 'emb_sup

In [16]:
# =========================
# MULTI-QUERY SANITY CHECK
# =========================

TEST_QUERIES = [
    # --- Sepsis / Shock-like ---
    "fever, low blood pressure, confusion",
    "fever, hypotension, altered mental status",
    "high fever, septic shock, hypotension",
    "fever, chills, low blood pressure, infection",
    
    # --- Infection but not shock ---
    "fever, cough, shortness of breath",
    "fever, sore throat, fatigue",
    "fever, diarrhea, abdominal pain",
    
    # --- Cardiovascular ---
    "low blood pressure, dizziness, fainting",
    "hypotension, chest pain, shortness of breath",
    
    # --- Neurologic ---
    "confusion, fever, headache, stiff neck",
    "altered mental status, fever",
    
    # --- Weak / underspecified ---
    "fever",
    "low blood pressure",
    "confusion",
]

for q in TEST_QUERIES:
    print("=" * 80)
    print("QUERY:", q)

    out = run_query(q, generate=False)

    print("\nSymptoms:")
    print(out["symptoms"])

    print("\nTop conditions:")
    for r in out["ranked_conditions"][:3]:
        print(
            f"{r['condition_concept_id']} | "
            f"graph={r['graph_score']:.3f} "
            f"emb={r['emb_score']:.1f} "
            f"final={r['final_score']:.3f}"
        )

QUERY: fever, low blood pressure, confusion

Symptoms:
[('UMLS:C0015967', 1.0, 'fever'), ('UMLS:C4304283', 1.0, 'low blood pressure')]

Top conditions:
UMLS:C3714514 | graph=0.724 emb=9.1 final=0.898
UMLS:C0034362 | graph=0.789 emb=7.5 final=0.892
UMLS:C0019345 | graph=0.749 emb=7.5 final=0.874
QUERY: fever, hypotension, altered mental status

Symptoms:
[('UMLS:C0015967', 1.0, 'fever'), ('UMLS:C0020649', 1.0, 'hypotension')]

Top conditions:
UMLS:C0221106 | graph=2.990 emb=2.8 final=1.357
UMLS:C0857353 | graph=0.980 emb=3.0 final=1.110
UMLS:C3714514 | graph=0.972 emb=2.7 final=1.053
QUERY: high fever, septic shock, hypotension

Symptoms:
[('UMLS:C0015967', 1.0, 'fever'), ('UMLS:C0020649', 1.0, 'hypotension')]

Top conditions:
UMLS:C0243026 | graph=1.278 emb=9.6 final=1.748
UMLS:C0036690 | graph=1.048 emb=9.6 final=1.706
UMLS:C0518988 | graph=0.972 emb=8.7 final=1.632
QUERY: fever, chills, low blood pressure, infection

Symptoms:
[('UMLS:C0015967', 1.0, 'fever'), ('UMLS:C4304283', 1.0, 

In [17]:
# =========================
# NOTEBOOK 05 — QUERY / HYBRID RETRIEVAL METRICS REPORT
# =========================
import time
import math
import numpy as np
import pandas as pd

print("=" * 90)
print("NOTEBOOK 05 METRICS — QUERY-TIME HYBRID RETRIEVAL")
print("=" * 90)

# -------------------------
# Helper: top-k sets
# -------------------------
def _topk_condition_set(df, col="condition_concept_id", k=10):
    if df is None or len(df) == 0 or col not in df.columns:
        return set()
    return set(df[col].head(k).tolist())

def _jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return np.nan
    return len(a & b) / len(a | b)

# -------------------------
# Measured runner
# -------------------------
def run_query_measured(query: str, top_graph=40, top_emb_chunks=30, top_emb_cond=40, top_final=5):
    t0 = time.perf_counter()

    symptoms = extract_symptom_concepts(query, known_symptoms)

    t_extract = time.perf_counter()

    GENERIC_SYMPTOMS = {
        "UMLS:C5139432",  # infection
    }
    symptoms = [s for s in symptoms if s[0] not in GENERIC_SYMPTOMS]
    symptom_ids = [cid for cid, _, _ in symptoms]

    SPECIFIC_SYMPTOMS = {
        "UMLS:C4304283",  # hypotension
        "UMLS:C5187432",  # altered mental status
    }

    specific_count = sum(1 for cid, _, _ in symptoms if cid in SPECIFIC_SYMPTOMS)
    use_graph = specific_count >= 1

    graph_df = graph_first_candidates(symptom_ids, top_k=top_graph)
    t_graph = time.perf_counter()

    emb_df = embedding_candidates(query, k_chunks=top_emb_chunks, k_cond=top_emb_cond)
    t_emb = time.perf_counter()

    ranked = merge_rerank(graph_df, emb_df, use_graph=use_graph, top_n=top_final)
    context = build_context(query, symptoms, ranked)
    t_done = time.perf_counter()

    final_evidence = []
    final_pmcids = set()
    for cond in context.get("conditions", []):
        ev = cond.get("evidence", [])
        final_evidence.extend(ev)
        final_pmcids.update([x["pmcid"] for x in ev if "pmcid" in x])

    graph_top10 = _topk_condition_set(graph_df, k=10)
    emb_top10 = _topk_condition_set(emb_df, k=10)
    final_top5 = _topk_condition_set(ranked, k=5)

    out = {
        "query": query,
        "n_symptoms_extracted": len(symptoms),
        "n_specific_symptoms": specific_count,
        "use_graph": bool(use_graph),
        "graph_candidates": int(len(graph_df)),
        "emb_candidates": int(len(emb_df)),
        "final_ranked_conditions": int(len(ranked)),
        "graph_top10_emb_top10_jaccard": _jaccard(graph_top10, emb_top10),
        "graph_nonempty": bool(len(graph_df) > 0),
        "emb_nonempty": bool(len(emb_df) > 0),
        "distinct_pmcids_in_final_evidence": int(len(final_pmcids)),
        "final_evidence_rows": int(len(final_evidence)),
        "extract_time_sec": t_extract - t0,
        "graph_time_sec": t_graph - t_extract,
        "emb_time_sec": t_emb - t_graph,
        "merge_context_time_sec": t_done - t_emb,
        "total_time_sec": t_done - t0,
        "top_final_condition_ids": list(final_top5),
        "symptoms": symptoms,
        "ranked_conditions": ranked,
        "context": context,
        "graph_df": graph_df,
        "emb_df": emb_df,
    }
    return out

# -------------------------
# Run suite
# -------------------------
if 'TEST_QUERIES' not in globals():
    TEST_QUERIES = [
        "fever, low blood pressure, confusion",
        "fever, cough, shortness of breath",
        "fever",
        "confusion",
    ]

rows = []
detailed = []

for q in TEST_QUERIES:
    res = run_query_measured(q)
    rows.append({
        "query": res["query"],
        "n_symptoms_extracted": res["n_symptoms_extracted"],
        "n_specific_symptoms": res["n_specific_symptoms"],
        "use_graph": res["use_graph"],
        "graph_candidates": res["graph_candidates"],
        "emb_candidates": res["emb_candidates"],
        "final_ranked_conditions": res["final_ranked_conditions"],
        "graph_top10_emb_top10_jaccard": res["graph_top10_emb_top10_jaccard"],
        "distinct_pmcids_in_final_evidence": res["distinct_pmcids_in_final_evidence"],
        "final_evidence_rows": res["final_evidence_rows"],
        "extract_time_sec": res["extract_time_sec"],
        "graph_time_sec": res["graph_time_sec"],
        "emb_time_sec": res["emb_time_sec"],
        "merge_context_time_sec": res["merge_context_time_sec"],
        "total_time_sec": res["total_time_sec"],
    })
    detailed.append(res)

metrics_df = pd.DataFrame(rows)

print("\nPer-query metrics:")
display(metrics_df)

print("\nAggregate metrics:")
agg = pd.DataFrame({
    "metric": [
        "queries_run",
        "queries_with_any_symptom",
        "queries_using_graph",
        "queries_with_graph_candidates",
        "queries_with_emb_candidates",
        "mean_symptoms_extracted",
        "mean_graph_emb_jaccard_top10",
        "mean_distinct_pmcids_in_final_evidence",
        "mean_final_evidence_rows",
        "mean_total_time_sec",
        "p95_total_time_sec",
    ],
    "value": [
        len(metrics_df),
        int((metrics_df["n_symptoms_extracted"] > 0).sum()),
        int(metrics_df["use_graph"].sum()),
        int((metrics_df["graph_candidates"] > 0).sum()),
        int((metrics_df["emb_candidates"] > 0).sum()),
        float(metrics_df["n_symptoms_extracted"].mean()),
        float(metrics_df["graph_top10_emb_top10_jaccard"].mean()),
        float(metrics_df["distinct_pmcids_in_final_evidence"].mean()),
        float(metrics_df["final_evidence_rows"].mean()),
        float(metrics_df["total_time_sec"].mean()),
        float(metrics_df["total_time_sec"].quantile(0.95)),
    ]
})
display(agg)

print("\nTop-1 condition snapshot per query:")
top1_rows = []
for res in detailed:
    ranked = res["ranked_conditions"]
    if len(ranked) == 0:
        top1_rows.append({
            "query": res["query"],
            "top1_condition": None,
            "graph_score": np.nan,
            "emb_score": np.nan,
            "final_score": np.nan,
        })
    else:
        r = ranked.iloc[0]
        top1_rows.append({
            "query": res["query"],
            "top1_condition": r["condition_concept_id"],
            "graph_score": float(r.get("graph_score", 0.0)),
            "emb_score": float(r.get("emb_score", 0.0)),
            "final_score": float(r.get("final_score", 0.0)),
        })

display(pd.DataFrame(top1_rows))

print("\nSaved log path:")
print(OUT_LOG.resolve() if 'OUT_LOG' in globals() else "N/A")

NOTEBOOK 05 METRICS — QUERY-TIME HYBRID RETRIEVAL

Per-query metrics:


,query,n_symptoms_extracted,n_specific_symptoms,use_graph,graph_candidates,emb_candidates,final_ranked_conditions,graph_top10_emb_top10_jaccard,distinct_pmcids_in_final_evidence,final_evidence_rows,extract_time_sec,graph_time_sec,emb_time_sec,merge_context_time_sec,total_time_sec
0,"fever, low blood pressure, confusion",2,1,True,40,34,5,0.666667,10,30,0.005011,0.010946,0.027090,0.188054,0.231101
1,"fever, hypotension, altered mental status",2,0,False,40,25,5,0.052632,16,28,0.004541,0.011969,0.029047,0.196083,0.241640
2,"high fever, septic shock, hypotension",2,0,False,40,40,5,0.052632,12,30,0.004962,0.013341,0.037079,0.189915,0.245297
3,"fever, chills, low blood pressure, infection",3,1,True,40,40,5,0.538462,10,30,0.004639,0.015045,0.031605,0.193438,0.244728
4,"fever, cough, shortness of breath",1,0,False,40,40,5,0.666667,11,30,0.004312,0.007901,0.036211,0.199811,0.248235
5,"fever, sore throat, fatigue",3,0,False,40,40,5,0.333333,9,30,0.003775,0.014373,0.037399,0.193548,0.249095
6,"fever, diarrhea, abdominal pain",4,0,False,40,40,5,0.428571,8,30,0.004150,0.014016,0.031553,0.194656,0.244375
7,"low blood pressure, dizziness, fainting",1,1,True,40,40,5,0.000000,15,25,0.004385,0.007458,0.035226,0.198251,0.245319
8,"hypotension, chest pain, shortness of breath",3,0,False,40,40,5,0.052632,14,30,0.004639,0.015183,0.028049,0.191446,0.239317
9,"confusion, fever, headache, stiff neck",3,0,False,40,40,5,0.000000,13,30,0.004134,0.013770,0.027467,0.206465,0.251836



Aggregate metrics:


,metric,value
0,queries_run,14.000000
1,queries_with_any_symptom,13.000000
2,queries_using_graph,4.000000
3,queries_with_graph_candidates,13.000000
4,queries_with_emb_candidates,14.000000
5,mean_symptoms_extracted,1.928571
6,mean_graph_emb_jaccard_top10,0.231146
7,mean_distinct_pmcids_in_final_evidence,11.357143
8,mean_final_evidence_rows,28.500000
9,mean_total_time_sec,0.240146



Top-1 condition snapshot per query:


,query,top1_condition,graph_score,emb_score,final_score
0,"fever, low blood pressure, confusion",UMLS:C3714514,0.724093,9.108963,0.898446
1,"fever, hypotension, altered mental status",UMLS:C0221106,2.989618,2.818501,1.357393
2,"high fever, septic shock, hypotension",UMLS:C0243026,1.278045,9.639975,1.748098
3,"fever, chills, low blood pressure, infection",UMLS:C3714514,0.872439,12.273286,1.018944
4,"fever, cough, shortness of breath",UMLS:C0034362,0.788657,15.380835,1.910253
5,"fever, sore throat, fatigue",UMLS:C0034362,1.380624,10.903359,1.833037
6,"fever, diarrhea, abdominal pain",UMLS:C0152517,3.408897,13.753313,2.208330
7,"low blood pressure, dizziness, fainting",UMLS:C0221106,0.107398,14.067226,0.624115
8,"hypotension, chest pain, shortness of breath",UMLS:C0221106,5.666564,6.333737,1.954333
9,"confusion, fever, headache, stiff neck",UMLS:C0034362,0.921888,13.662358,1.872493



Saved log path:
D:\Pictures\pmc_graphrag\data\parsed\query_logs.jsonl
